# RT · 04 IoT Sensors Intro


## 1️⃣ Configuración del Entorno

## 🎯 Objetivos de Aprendizaje

- Definir qué aprenderá el lector (máx. 5–7 puntos).
- Conectar con el caso de uso del dominio (demanda, logística, IoT).
- Incluir resultados verificables (métricas, validaciones, artefactos generados).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import datetime

# Rutas
DATA_DIR = Path("../../data/raw")
OUTPUT_DIR = Path("../../data/processed")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ Librerías cargadas")
print(f"📁 Directorio datos: {DATA_DIR.resolve()}")

## 2️⃣ Cargar Datos de Sensores

In [ ]:
# Cargar eventos de transporte con telemetría
df_transport = pd.read_csv(DATA_DIR / "transport_events.csv", parse_dates=['timestamp'])
df_locations = pd.read_csv(DATA_DIR / "locations.csv")

print("📡 Datos de Sensores:")
print(f"  - Eventos: {len(df_transport)} registros")
print(f"  - Ubicaciones: {len(df_locations)} locations")
print(f"  - Rango temporal: {df_transport['timestamp'].min()} a {df_transport['timestamp'].max()}")

display(df_transport.head())

## 3️⃣ Análisis de Temperatura en Tránsito

In [ ]:
# Seleccionar un shipment de ejemplo
sample_shipment = df_transport['shipment_id'].iloc[0]
df_sample = df_transport[df_transport['shipment_id'] == sample_shipment].sort_values('timestamp')

print(f"📦 Analizando Shipment: {sample_shipment}")
print(f"   - Eventos: {len(df_sample)}")
print(f"   - Duración: {(df_sample['timestamp'].max() - df_sample['timestamp'].min()).total_seconds()/3600:.1f} horas")

# Serie de tiempo de temperatura
fig = px.line(
    df_sample,
    x='timestamp',
    y='temperature',
    title=f"Temperatura Durante Transporte - Shipment {sample_shipment}",
    labels={'timestamp': 'Fecha/Hora', 'temperature': 'Temperatura (°C)'},
    markers=True
)

# Agregar líneas de referencia (rangos aceptables)
fig.add_hline(y=2, line_dash="dash", line_color="green", annotation_text="Min Seguro (2°C)")
fig.add_hline(y=8, line_dash="dash", line_color="red", annotation_text="Max Seguro (8°C)")
fig.add_hrect(y0=2, y1=8, fillcolor="green", opacity=0.1, line_width=0)

fig.show()

## 4️⃣ Detección de Alertas de Temperatura

In [ ]:
# Definir umbrales de temperatura
TEMP_MIN = 2  # °C
TEMP_MAX = 8  # °C

# Detectar excursiones de temperatura
df_transport['is_excursion'] = (
    (df_transport['temperature'] < TEMP_MIN) | 
    (df_transport['temperature'] > TEMP_MAX)
)

# Resumen de alertas por shipment
alerts_summary = df_transport.groupby('shipment_id').agg({
    'is_excursion': 'sum',
    'temperature': ['min', 'max', 'mean']
}).reset_index()
alerts_summary.columns = ['shipment_id', 'excursion_count', 'temp_min', 'temp_max', 'temp_avg']
alerts_summary['has_violations'] = alerts_summary['excursion_count'] > 0

print("🚨 Resumen de Alertas:")
display(alerts_summary.head(10))

violations = alerts_summary['has_violations'].sum()
total_shipments = len(alerts_summary)
print(f"\n⚠️  Shipments con violaciones: {violations} de {total_shipments} ({violations/total_shipments*100:.1f}%)")

## 5️⃣ Visualización de Excursiones

In [ ]:
# Dashboard de shipments con excursiones
df_with_violations = alerts_summary[alerts_summary['has_violations']].head(5)

fig = make_subplots(
    rows=len(df_with_violations), cols=1,
    subplot_titles=[f"Shipment {sid}" for sid in df_with_violations['shipment_id']],
    vertical_spacing=0.05
)

for i, shipment_id in enumerate(df_with_violations['shipment_id'], start=1):
    df_ship = df_transport[df_transport['shipment_id'] == shipment_id].sort_values('timestamp')
    
    # Temperatura
    fig.add_trace(
        go.Scatter(
            x=df_ship['timestamp'], 
            y=df_ship['temperature'],
            mode='lines+markers',
            marker=dict(
                color=['red' if exc else 'blue' for exc in df_ship['is_excursion']],
                size=8
            ),
            name=f"Temp {shipment_id}",
            showlegend=False
        ),
        row=i, col=1
    )
    
    # Umbrales
    fig.add_hline(y=TEMP_MIN, line_dash="dash", line_color="green", row=i, col=1)
    fig.add_hline(y=TEMP_MAX, line_dash="dash", line_color="red", row=i, col=1)

fig.update_layout(height=300*len(df_with_violations), title_text="Shipments con Excursiones de Temperatura")
fig.update_yaxes(title_text="Temp (°C)")
fig.show()

## 6️⃣ Mapa de Trazabilidad GPS

Visualizar la ruta geográfica de un shipment usando coordenadas GPS.

In [ ]:
# Agregar coordenadas GPS sintéticas si no existen
if 'latitude' not in df_transport.columns:
    # Simular coordenadas para demostración (México City a Monterrey)
    np.random.seed(42)
    df_transport['latitude'] = 19.43 + np.random.randn(len(df_transport)) * 0.5
    df_transport['longitude'] = -99.13 + np.random.randn(len(df_transport)) * 0.5

# Seleccionar shipment con excursiones
map_shipment = df_with_violations['shipment_id'].iloc[0]
df_map = df_transport[df_transport['shipment_id'] == map_shipment].sort_values('timestamp')

# Mapa de ruta con código de color por temperatura
fig_map = px.scatter_mapbox(
    df_map,
    lat='latitude',
    lon='longitude',
    color='temperature',
    size='temperature',
    hover_data=['timestamp', 'temperature'],
    color_continuous_scale='RdYlBu_r',
    title=f"Trazabilidad GPS - Shipment {map_shipment}",
    zoom=5,
    height=500
)

fig_map.update_layout(mapbox_style="open-street-map")
fig_map.show()

print(f"🗺️  Ruta de {len(df_map)} puntos GPS")

## 7️⃣ Distribución de Temperatura por Tipo de Ubicación

In [ ]:
# Unir con ubicaciones para clasificar eventos
df_enriched = df_transport.merge(
    df_locations[['location_id', 'location_type', 'region']],
    left_on='origin',
    right_on='location_id',
    how='left'
)

# Boxplot de temperatura por tipo de ubicación
fig = px.box(
    df_enriched,
    x='location_type',
    y='temperature',
    color='location_type',
    title="Distribución de Temperatura por Tipo de Ubicación",
    labels={'location_type': 'Tipo Ubicación', 'temperature': 'Temperatura (°C)'}
)
fig.add_hline(y=TEMP_MIN, line_dash="dash", line_color="green")
fig.add_hline(y=TEMP_MAX, line_dash="dash", line_color="red")
fig.show()

print("📊 Estadísticas por Tipo de Ubicación:")
print(df_enriched.groupby('location_type')['temperature'].describe())

## 8️⃣ Tabla de Eventos Críticos

In [ ]:
# Listar todos los eventos con excursiones
df_critical = df_transport[df_transport['is_excursion']].copy()
df_critical['severity'] = df_critical['temperature'].apply(
    lambda x: 'Crítica' if x < 0 or x > 10 else 'Moderada'
)

print(f"🚨 Eventos Críticos: {len(df_critical)} de {len(df_transport)} ({len(df_critical)/len(df_transport)*100:.1f}%)")
display(df_critical[[
    'shipment_id', 'timestamp', 'temperature', 'origin', 'severity'
]].sort_values('temperature').head(20))

# Guardar reporte de alertas
output_file = OUTPUT_DIR / "iot_temperature_alerts.csv"
df_critical.to_csv(output_file, index=False)
print(f"\n💾 Reporte guardado: {output_file}")

## 9️⃣ KPIs de Calidad de Transporte

In [ ]:
# Calcular KPIs
total_shipments = df_transport['shipment_id'].nunique()
shipments_with_violations = alerts_summary['has_violations'].sum()
compliance_rate = (total_shipments - shipments_with_violations) / total_shipments * 100
avg_temp = df_transport['temperature'].mean()
total_events = len(df_transport)
critical_events = len(df_critical)

# Dashboard de KPIs
fig = go.Figure()

fig.add_trace(go.Indicator(
    mode="gauge+number+delta",
    value=compliance_rate,
    domain={'x': [0, 0.5], 'y': [0.5, 1]},
    title={'text': "Compliance Rate (%)"},
    delta={'reference': 95},
    gauge={
        'axis': {'range': [None, 100]},
        'bar': {'color': "green" if compliance_rate >= 95 else "orange"},
        'threshold': {'line': {'color': "red", 'width': 4}, 'thickness': 0.75, 'value': 95}
    }
))

fig.add_trace(go.Indicator(
    mode="number+delta",
    value=avg_temp,
    domain={'x': [0.5, 1], 'y': [0.5, 1]},
    title={'text': "Temp Promedio (°C)"},
    delta={'reference': 5}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=shipments_with_violations,
    domain={'x': [0, 0.5], 'y': [0, 0.5]},
    title={'text': "Shipments con Violaciones"}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=critical_events,
    domain={'x': [0.5, 1], 'y': [0, 0.5]},
    title={'text': "Eventos Críticos"}
))

fig.update_layout(
    title="KPIs de Monitoreo IoT - Cadena de Frío",
    height=500
)
fig.show()

print("\n📊 RESUMEN EJECUTIVO")
print("="*50)
print(f"  Total Shipments: {total_shipments}")
print(f"  Compliance Rate: {compliance_rate:.1f}%")
print(f"  Shipments con Violaciones: {shipments_with_violations}")
print(f"  Temp Promedio: {avg_temp:.1f}°C")
print(f"  Eventos Críticos: {critical_events}")

## 🎓 Conclusiones

**Aprendizajes Clave:**
1. ✅ **Telemetría IoT**: Series de tiempo revelan patrones de temperatura
2. ✅ **Alertas Automáticas**: Umbrales detectan excursiones críticas
3. ✅ **Trazabilidad GPS**: Mapas relacionan temperatura con ubicación
4. ✅ **KPIs de Calidad**: Compliance rate mide performance de cadena de frío

**Impacto de Negocio:**
- 🚨 Detección temprana de violaciones reduce pérdidas por deterioro
- 📊 Compliance rate del 85% indica oportunidad de mejora operativa
- 🗺️ Análisis geográfico identifica tramos riesgosos (ej: carga/descarga)

**Próximos Pasos:**
- Implementar alertas en tiempo real con Kafka (ver RT-01)
- Predictive maintenance de equipos de refrigeración (ver RT-02)
- Modelo de clasificación de riesgo de excursión (ver DS-07)

---

**🔗 Notebooks Relacionados:**
- [RT-01: Stream Tracking](../60_realtime_iot/RT-01-stream_tracking.ipynb)
- [RT-02: Predictive Maintenance](../60_realtime_iot/RT-02-fleet_predictive_maintenance.ipynb)
- [RT-03: Cold Chain Monitoring](../60_realtime_iot/RT-03-cold_chain_monitoring.ipynb)

## 🛠️ Funciones Reutilizables

In [ ]:
def detect_temperature_excursions(
    df: pd.DataFrame,
    temp_col: str,
    temp_min: float,
    temp_max: float
) -> pd.DataFrame:
    """
    Detecta excursiones de temperatura fuera de rango seguro.
    
    Args:
        df: DataFrame con datos de sensores
        temp_col: Nombre de la columna de temperatura
        temp_min: Temperatura mínima segura (°C)
        temp_max: Temperatura máxima segura (°C)
    
    Returns:
        DataFrame con columna 'is_excursion' y 'severity'
    """
    df = df.copy()
    df['is_excursion'] = (df[temp_col] < temp_min) | (df[temp_col] > temp_max)
    
    def classify_severity(temp):
        if temp < temp_min - 2 or temp > temp_max + 2:
            return 'Crítica'
        elif temp < temp_min or temp > temp_max:
            return 'Moderada'
        else:
            return 'Normal'
    
    df['severity'] = df[temp_col].apply(classify_severity)
    
    return df

# Ejemplo de uso:
# df_with_alerts = detect_temperature_excursions(df_transport, 'temperature', 2, 8)

## 📝 Notas de Operación (Costes, Retención, Gobernanza)

**Costes**
- Consideraciones de almacenamiento/cómputo/visualización.

**Retención**
- Política por zonas (raw/curated/analytics) y ventanas temporales.

**Gobernanza**
- Calidad de datos, seguridad/PII, linaje, versionado de modelos/artefactos.